# 🏋️ Esteira de Aprendizado de Máquina
## World Gym & Fitness Trends (2000–2026)

**Objetivo:** Classificar o nível de engajamento em academias com base em tendências globais de fitness.

**Fonte dos dados:** Kaggle — World Gym & Fitness Trends (2000–2026)

**Problema:** Classificação multiclasse — prever a categoria de crescimento de membros (`membership_growth_category`)

---

## 0. Instalação e Importação de Bibliotecas

In [ ]:
# Instalação (caso necessário)
# !pip install pandas numpy scikit-learn matplotlib seaborn kagglehub

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    confusion_matrix, accuracy_score,
    classification_report, ConfusionMatrixDisplay
)

# Seed para reprodutibilidade
SEED = 42
np.random.seed(SEED)

print('✅ Bibliotecas importadas com sucesso!')

## 1. Carregamento do Dataset

> **Fonte:** [Kaggle – World Gym & Fitness Trends (2000–2026)](https://www.kaggle.com/datasets/)</br>
> O dataset contém dados globais sobre academias, número de membros, tendências de fitness e indicadores econômicos ao longo de 26 anos.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# OPÇÃO A: Download via kagglehub (requer kaggle.json configurado)
# ─────────────────────────────────────────────────────────────────────────────
# import kagglehub
# path = kagglehub.dataset_download('your-username/world-gym-fitness-trends')
# df = pd.read_csv(f'{path}/gym_fitness_trends.csv')

# ─────────────────────────────────────────────────────────────────────────────
# OPÇÃO B: Carregar CSV baixado manualmente do Kaggle
# ─────────────────────────────────────────────────────────────────────────────
# df = pd.read_csv('gym_fitness_trends.csv')

# ─────────────────────────────────────────────────────────────────────────────
# OPÇÃO C: Dataset sintético fiel à estrutura real (fallback para demonstração)
# Use esta opção se não tiver o arquivo CSV disponível
# ─────────────────────────────────────────────────────────────────────────────

np.random.seed(SEED)
n = 1500

years      = np.random.randint(2000, 2027, n)
countries  = np.random.choice(
    ['USA','Brazil','Germany','UK','Australia','Japan','India','France','Canada','China'], n)
gym_types  = np.random.choice(
    ['Commercial','Boutique','CrossFit','Yoga Studio','Corporate'], n)
memberships        = np.random.randint(100, 15000, n)
avg_monthly_fee    = np.round(np.random.uniform(15, 200, n), 2)
trainers_per_gym   = np.random.randint(1, 40, n)
online_classes_pct = np.round(np.random.uniform(0, 100, n), 1)
retention_rate     = np.round(np.random.uniform(40, 99, n), 1)
gdp_per_capita     = np.round(np.random.uniform(5000, 80000, n), 0)
urbanization_pct   = np.round(np.random.uniform(30, 98, n), 1)
obesity_rate       = np.round(np.random.uniform(5, 40, n), 1)

# Crescimento percentual anual de membros
membership_growth = (
    0.002 * (years - 2000)
    + 0.00001 * gdp_per_capita
    + 0.003  * retention_rate
    - 0.001  * avg_monthly_fee
    + np.random.normal(0, 0.05, n)
)

# Variável-alvo: categoria de crescimento
def categorize_growth(g):
    if g < 0.1:   return 'Baixo'
    elif g < 0.35: return 'Médio'
    else:          return 'Alto'

growth_label = [categorize_growth(g) for g in membership_growth]

df = pd.DataFrame({
    'year':               years,
    'country':            countries,
    'gym_type':           gym_types,
    'memberships':        memberships,
    'avg_monthly_fee':    avg_monthly_fee,
    'trainers_per_gym':   trainers_per_gym,
    'online_classes_pct': online_classes_pct,
    'retention_rate':     retention_rate,
    'gdp_per_capita':     gdp_per_capita,
    'urbanization_pct':   urbanization_pct,
    'obesity_rate':       obesity_rate,
    'membership_growth':  np.round(membership_growth, 4),
    'growth_category':    growth_label   # ← variável-alvo
})

print(f'✅ Dataset carregado: {df.shape[0]} linhas × {df.shape[1]} colunas')
df.head()

## 2. Estatísticas Descritivas Gerais

In [ ]:
print('=== INFORMAÇÕES GERAIS ===')
df.info()

In [ ]:
print('\n=== ESTATÍSTICAS DESCRITIVAS (variáveis numéricas) ===')
df.describe().round(2)

In [ ]:
print('\n=== VALORES NULOS POR COLUNA ===')
print(df.isnull().sum())

print('\n=== DISTRIBUIÇÃO DA VARIÁVEL-ALVO ===')
print(df['growth_category'].value_counts())
print()
print(df['growth_category'].value_counts(normalize=True).mul(100).round(1).astype(str) + '%')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Distribuições das Variáveis Numéricas', fontsize=16, fontweight='bold', y=1.01)

cols = ['memberships','avg_monthly_fee','retention_rate',
        'gdp_per_capita','online_classes_pct','obesity_rate']
colors = ['#2196F3','#4CAF50','#FF9800','#9C27B0','#F44336','#009688']

for ax, col, color in zip(axes.flat, cols, colors):
    ax.hist(df[col], bins=30, color=color, alpha=0.8, edgecolor='white')
    ax.set_title(col.replace('_',' ').title(), fontsize=11)
    ax.set_xlabel('')
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('distribuicoes.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Histogramas plotados')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribuição da variável-alvo
colors_target = ['#4CAF50','#FF9800','#F44336']
vc = df['growth_category'].value_counts()
axes[0].bar(vc.index, vc.values, color=colors_target, edgecolor='white', linewidth=1.5)
axes[0].set_title('Distribuição da Variável-Alvo\n(growth_category)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Contagem')
for i, (k, v) in enumerate(vc.items()):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

# Correlação numérica
num_cols = df.select_dtypes(include='number').columns.tolist()
corr = df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, ax=axes[1], annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, linewidths=0.5, annot_kws={'size':7})
axes[1].set_title('Mapa de Correlação', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('exploratorio.png', dpi=120, bbox_inches='tight')
plt.show()

## 3. Transformações nas Colunas (features)

Aplicamos as seguintes transformações:
- **Codificação de variáveis categóricas** (`country`, `gym_type`) via `LabelEncoder`
- **Transformação logarítmica** em `gdp_per_capita` e `memberships` para reduzir assimetria
- **Normalização** das variáveis numéricas via `StandardScaler`
- **Criação de feature** `fee_per_trainer` (razão entre taxa mensal e número de instrutores)

In [ ]:
df_proc = df.copy()

# ── 3.1 Codificação de categóricas ──────────────────────────────────────────
le_country  = LabelEncoder()
le_gymtype  = LabelEncoder()
le_target   = LabelEncoder()

df_proc['country_enc']  = le_country.fit_transform(df_proc['country'])
df_proc['gym_type_enc'] = le_gymtype.fit_transform(df_proc['gym_type'])
df_proc['target']       = le_target.fit_transform(df_proc['growth_category'])

print('Classes codificadas:', dict(zip(le_target.classes_, le_target.transform(le_target.classes_))))

# ── 3.2 Transformação logarítmica ───────────────────────────────────────────
df_proc['log_gdp']         = np.log1p(df_proc['gdp_per_capita'])
df_proc['log_memberships'] = np.log1p(df_proc['memberships'])

# ── 3.3 Feature engineering ─────────────────────────────────────────────────
df_proc['fee_per_trainer'] = df_proc['avg_monthly_fee'] / (df_proc['trainers_per_gym'] + 1)

print('\n✅ Transformações de colunas aplicadas.')
df_proc[['log_gdp','log_memberships','fee_per_trainer','target']].head()

## 4. Transformações nas Linhas

- **Remoção de outliers** em `avg_monthly_fee` via IQR (linhas cujo valor está além de 3×IQR)
- **Remoção de duplicatas**

In [ ]:
before = len(df_proc)

# ── 4.1 Remoção de duplicatas ────────────────────────────────────────────────
df_proc.drop_duplicates(inplace=True)
print(f'Após remover duplicatas: {len(df_proc)} linhas (removidas: {before - len(df_proc)})')

# ── 4.2 Remoção de outliers via IQR (3×IQR) ──────────────────────────────────
def remove_outliers_iqr(dataframe, column, factor=3.0):
    Q1 = dataframe[column].quantile(0.25)
    Q3 = dataframe[column].quantile(0.75)
    IQR = Q3 - Q1
    mask = (dataframe[column] >= Q1 - factor * IQR) & (dataframe[column] <= Q3 + factor * IQR)
    return dataframe[mask]

after_dup = len(df_proc)
df_proc = remove_outliers_iqr(df_proc, 'avg_monthly_fee')
df_proc = remove_outliers_iqr(df_proc, 'membership_growth')
print(f'Após remover outliers:   {len(df_proc)} linhas (removidas: {after_dup - len(df_proc)})')

df_proc.reset_index(drop=True, inplace=True)
print('\n✅ Transformações de linhas concluídas.')

## 5. Divisão em Subconjuntos: Treino / Validação / Teste

| Subconjunto | Proporção | Uso |
|-------------|-----------|-----|
| Treino      | 70%       | Ajuste dos parâmetros do modelo |
| Validação   | 15%       | Ajuste de hiperparâmetros |
| Teste       | 15%       | Avaliação final imparcial |

In [ ]:
FEATURES = [
    'year', 'country_enc', 'gym_type_enc',
    'log_memberships', 'avg_monthly_fee', 'trainers_per_gym',
    'online_classes_pct', 'retention_rate', 'log_gdp',
    'urbanization_pct', 'obesity_rate', 'fee_per_trainer'
]
TARGET = 'target'

X = df_proc[FEATURES]
y = df_proc[TARGET]

# 70% treino | 30% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y)

# 50% de 30% = 15% val | 15% teste
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp)

# Normalização (fit apenas no treino)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)
X_test_sc  = scaler.transform(X_test)

print(f'Treino:    {X_train_sc.shape[0]} amostras ({X_train_sc.shape[0]/len(X)*100:.1f}%)')
print(f'Validação: {X_val_sc.shape[0]} amostras ({X_val_sc.shape[0]/len(X)*100:.1f}%)')
print(f'Teste:     {X_test_sc.shape[0]} amostras ({X_test_sc.shape[0]/len(X)*100:.1f}%)')
print('\n✅ Divisão e normalização concluídas.')

## 6. Treinamento do Modelo — Random Forest Classifier

Utilizamos **Random Forest** por sua robustez, interpretabilidade e bom desempenho em dados tabulares.
O ajuste fino de hiperparâmetros usa o conjunto de **validação**.

In [ ]:
# ── Treinamento ──────────────────────────────────────────────────────────────
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight='balanced',
    random_state=SEED,
    n_jobs=-1
)
model.fit(X_train_sc, y_train)

# ── Avaliação no conjunto de validação ──────────────────────────────────────
y_val_pred = model.predict(X_val_sc)
val_acc    = accuracy_score(y_val, y_val_pred)

print(f'✅ Modelo treinado!')
print(f'\n📊 Acurácia no conjunto de VALIDAÇÃO: {val_acc:.4f} ({val_acc*100:.2f}%)')
print('\nRelatório de Classificação (Validação):')
print(classification_report(y_val, y_val_pred,
                              target_names=le_target.classes_))

## 7. Avaliação Final — Conjunto de Teste
### 7.1 Acurácia e Relatório de Classificação

In [ ]:
y_test_pred = model.predict(X_test_sc)
test_acc    = accuracy_score(y_test, y_test_pred)

print(f'🎯 Acurácia no conjunto de TESTE: {test_acc:.4f} ({test_acc*100:.2f}%)')
print('\nRelatório de Classificação (Teste):')
print(classification_report(y_test, y_test_pred,
                              target_names=le_target.classes_))

### 7.2 Matriz de Confusão

In [ ]:
cm = confusion_matrix(y_test, y_test_pred)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Matriz absoluta ──────────────────────────────────────────────────────────
disp = ConfusionMatrixDisplay(cm, display_labels=le_target.classes_)
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Matriz de Confusão\n(contagens absolutas)', fontsize=12, fontweight='bold')

# ── Matriz normalizada ───────────────────────────────────────────────────────
cm_norm = cm.astype(float) / cm.sum(axis=1)[:, np.newaxis]
disp2   = ConfusionMatrixDisplay(cm_norm.round(2), display_labels=le_target.classes_)
disp2.plot(ax=axes[1], colorbar=False, cmap='Greens')
axes[1].set_title('Matriz de Confusão\n(proporções por classe)', fontsize=12, fontweight='bold')

plt.suptitle(f'Acurácia no Teste: {test_acc*100:.2f}%', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Matriz de confusão gerada.')

### 7.3 Importância das Features

In [ ]:
importances = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(importances.index, importances.values,
               color=plt.cm.viridis(importances.values / importances.max()),
               edgecolor='white')
ax.set_title('Importância das Features\n(Random Forest)', fontsize=13, fontweight='bold')
ax.set_xlabel('Importância relativa')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()

## 8. Predição com o Modelo Implantado

Simulamos um cenário real: uma academia boutique no Brasil em 2025.

In [ ]:
# ── Novo registro para predição ──────────────────────────────────────────────
novo_exemplo = {
    'year':               2025,
    'country':            'Brazil',
    'gym_type':           'Boutique',
    'memberships':        800,
    'avg_monthly_fee':    120.0,
    'trainers_per_gym':   8,
    'online_classes_pct': 65.0,
    'retention_rate':     82.0,
    'gdp_per_capita':     9500.0,
    'urbanization_pct':   87.0,
    'obesity_rate':       22.3,
}

print('📋 Dados de entrada (nova academia):')
for k, v in novo_exemplo.items():
    print(f'   {k:<22} = {v}')

# ── Pré-processamento do novo exemplo ────────────────────────────────────────
country_enc_val  = le_country.transform([novo_exemplo['country']])[0]
gym_type_enc_val = le_gymtype.transform([novo_exemplo['gym_type']])[0]

X_novo = pd.DataFrame([{
    'year':               novo_exemplo['year'],
    'country_enc':        country_enc_val,
    'gym_type_enc':       gym_type_enc_val,
    'log_memberships':    np.log1p(novo_exemplo['memberships']),
    'avg_monthly_fee':    novo_exemplo['avg_monthly_fee'],
    'trainers_per_gym':   novo_exemplo['trainers_per_gym'],
    'online_classes_pct': novo_exemplo['online_classes_pct'],
    'retention_rate':     novo_exemplo['retention_rate'],
    'log_gdp':            np.log1p(novo_exemplo['gdp_per_capita']),
    'urbanization_pct':   novo_exemplo['urbanization_pct'],
    'obesity_rate':       novo_exemplo['obesity_rate'],
    'fee_per_trainer':    novo_exemplo['avg_monthly_fee'] / (novo_exemplo['trainers_per_gym'] + 1),
}])

X_novo_sc = scaler.transform(X_novo)

# ── Predição ──────────────────────────────────────────────────────────────────
pred_encoded = model.predict(X_novo_sc)[0]
pred_label   = le_target.inverse_transform([pred_encoded])[0]
pred_proba   = model.predict_proba(X_novo_sc)[0]

print('\n' + '='*50)
print(f'🔮 PREDIÇÃO: Categoria de crescimento = [{pred_label.upper()}]')
print('='*50)
print('\n📊 Probabilidades por classe:')
for cls, prob in zip(le_target.classes_, pred_proba):
    bar = '█' * int(prob * 30)
    print(f'   {cls:<8} | {bar:<30} | {prob*100:5.1f}%')

## 9. Resumo Final

In [ ]:
print('=' * 60)
print('          RESUMO DA ESTEIRA DE APRENDIZADO DE MÁQUINA')
print('=' * 60)
print(f'  Dataset          : World Gym & Fitness Trends (2000-2026)')
print(f'  Problema         : Classificação multiclasse')
print(f'  Variável-alvo    : growth_category (Alto/Médio/Baixo)')
print(f'  Amostras totais  : {len(df_proc)}')
print(f'  Features usadas  : {len(FEATURES)}')
print(f'  Algoritmo        : Random Forest Classifier')
print(f'  Acurácia treino  : {accuracy_score(y_train, model.predict(X_train_sc))*100:.2f}%')
print(f'  Acurácia validação: {val_acc*100:.2f}%')
print(f'  Acurácia teste   : {test_acc*100:.2f}%')
print(f'  Predição exemplo : {pred_label} (confiança: {max(pred_proba)*100:.1f}%)')
print('=' * 60)
print('✅ Esteira concluída com sucesso!')